# T03 — Multimodal RAG: Text and Images

A RAG pipeline that retrieves both text and images from one PDF:

1. Parse the PDF into text and image chunks
2. Embed text with OpenAI, images with a vision-language model
3. Store both in separate ChromaDB collections
4. Retrieve from both with one composite retriever
5. Pass text and images to the LLM for a grounded answer

> **Optional, not covered in the workshop.** Sections 1 and 2 run anywhere. From section 3 on, images are embedded with `Qwen/Qwen3-VL-Embedding-2B`, which downloads about 4 GB and needs roughly 8 GB of free RAM. In a ~7 GB session the kernel will likely crash there; run it on a machine with more memory.

## Setup

Requires `OPENAI_API_KEY` (locally: in `.env.local` at the repo root).

In [ ]:
import base64
import io
import sys

import matplotlib.pyplot as plt
from loguru import logger

from conversational_toolkit.agents.rag import RAG
from conversational_toolkit.chunking.base import Chunk
from conversational_toolkit.chunking.pdf_chunker import PDFChunker
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.retriever.vectorstore_retriever import (
    CompositeVectorStoreRetriever,
    VectorStoreRetriever,
)
from ki_kompass_rag.baseline_rag import (
    DATA_DIR,
    DB_DIR,
    EMBEDDING_MODEL,
    ask,
    build_llm,
    build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")


def show_image(chunk: Chunk, size: int = 5) -> None:
    image_format = chunk.mime_type.split("/")[1]
    image = plt.imread(io.BytesIO(base64.b64decode(chunk.content)), format=image_format)
    plt.figure(figsize=(size, size))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

## 1. Parse the Document

`PDFChunker.make_chunks()` returns one text chunk per section and, by default, one chunk per embedded picture. Images are stored as base64-encoded PNG in `content`, so they fit the same `Chunk` structure.

In [ ]:
chunks = PDFChunker(do_ocr=False).make_chunks(
    str(DATA_DIR / "EPD_cardboard_redbox_cartonpallet.pdf")
)
text_chunks = [c for c in chunks if c.mime_type.startswith("text/")]
image_chunks = [c for c in chunks if c.mime_type.startswith("image/")]
print(f"{len(text_chunks)} text chunks, {len(image_chunks)} images")

for chunk in image_chunks:
    print(chunk.title)
    show_image(chunk, size=3)

## 2. Embed Text

Text chunks go into their own collection, embedded with `text-embedding-3-small` as in T01.

In [ ]:
text_embedding_model = OpenAIEmbeddings(EMBEDDING_MODEL)
text_vector_store = await build_vector_store(
    text_chunks, text_embedding_model, db_path=DB_DIR / "t03_text", reset=True
)

query = "What is the amount of PBT/vPvB substances?"
results = await VectorStoreRetriever(
    text_embedding_model, text_vector_store, top_k=3
).retrieve(query)
for r in results:
    print(f"{r.score:.3f}  {r.title}")
print(f"\n{results[0].content}")

## 3. Embed Images

> Needs about 8 GB of free RAM. Sections 3 to 6 depend on this model.

`Qwen3VLEmbeddings` maps images and text into the same vector space, so images can be found with a text query: `get_embeddings` encodes the image chunks, `get_text_embeddings` the query.

In [ ]:
from conversational_toolkit.embeddings.qwen_vl import Qwen3VLEmbeddings

image_embedding_model = Qwen3VLEmbeddings()
image_vector_store = await build_vector_store(
    image_chunks, image_embedding_model, db_path=DB_DIR / "t03_image", reset=True
)

In [ ]:
query = "The logo of a company"  # or "A carton box on a table", "The EPD logo"

query_embedding = await image_embedding_model.get_text_embeddings(query)
for r in await image_vector_store.get_chunks_by_embedding(query_embedding[0], top_k=3):
    print(f"{r.score:.3f}  {r.title}")
    show_image(r, size=4)

## 4. Composite Retrieval

`CompositeVectorStoreRetriever` queries both stores in parallel and concatenates the results: here the top 3 text chunks and the top image.

In [ ]:
composite_retriever = CompositeVectorStoreRetriever(
    embedding_models=[text_embedding_model, image_embedding_model],
    vector_stores=[text_vector_store, image_vector_store],
    top_k=[3, 1],
)

results = await composite_retriever.retrieve(
    "What is the amount of PBT/vPvB substances?"
)
for r in results:
    print(f"{r.mime_type}  {r.title}")
    if r.mime_type.startswith("image/"):
        show_image(r)

## 5. Send an Image to the LLM

A multimodal message holds a text block and an image block (base64 PNG). `generate` returns the full answer, `generate_stream` yields it piece by piece.

In [ ]:
llm = build_llm("openai")
messages = [
    LLMMessage(
        role=Roles.USER,
        content=[
            MessageContent(type="text", text="What is in this image?"),
            MessageContent(type="image", image_url=results[-1].content),
        ],
    )
]

answer = await llm.generate(messages)
print(answer.content[0].text)

In [ ]:
async for chunk in llm.generate_stream(messages):
    print("".join(c.text or "" for c in chunk.content), end="")

## 6. Full RAG Pipeline

The `RAG` agent with the composite retriever sends the retrieved text and images to the LLM. `ask()` prints the answer and the sources; retrieved images are shown below it.

In [ ]:
agent = RAG(
    llm=llm,
    utility_llm=llm,
    system_prompt=(
        "Answer questions about the document using only the retrieved text and "
        "images. If they do not contain the answer, say you don't know."
    ),
    retrievers=[composite_retriever],
)

for query in [
    "What is the amount of PBT/vPvB substances? Short answer.",
    "What is the percentage of glue in the product? Short answer.",
]:
    print(f"\n>>> {query}")
    answer = await ask(agent, query)
    for source in answer.sources:
        if source.mime_type.startswith("image/"):
            show_image(source, size=4)